Plan

For one image
- load
- detect grid points
- pixels to mm conversion
- sam2 -> berry mask
- main axis: position and length (see Eurbee2026)
- perpendicular and through middle: 2nd axis: position and length

For multiple images of the same berry
- main and second axes: lengths
- take smallest and largest 2nd axis -> 2nd and 3rd axis of 3D berry
- volume of berry with ellipsoid formula

For all berries
- volume

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
import numpy as np
import itertools
import torch
import torchvision
import sys
import os
import shutil
from pathlib import Path
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.morphology import convex_hull_image


In [ ]:
def make_mask(img, predictor, step = 500, visualize = False):
    
    # Generate query points for SAM
    queries = list(itertools.product(
        np.arange(step//2, img.shape[1], step),
        np.arange(step//2, img.shape[0], step), 
    ))
    
    # Apply SAM2
    predictor.set_image(img)
    candidates = []
    
    for point in queries:
        input_point = np.array([point])
        input_label = np.array([1])
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True,
        )
    
        # Some selection
        for mask in masks:
    
            # Size
            mask_size_min = 100000
            mask_size_max = 1000000
            mask_size = np.sum(mask)
            if mask_size < mask_size_min or mask_size > mask_size_max:
                break
        
            # Centeredness
            mask_cx_min = mask.shape[0]*0.25
            mask_cx_max = mask.shape[0]*0.75
            mask_cx = np.mean(np.where(mask)[0])
            if mask_cx < mask_cx_min or mask_cx > mask_cx_max:
                break
            
            mask_cy_min = mask.shape[1]*0.25
            mask_cy_max = mask.shape[1]*0.75
            mask_cy = np.mean(np.where(mask)[1])
            if mask_cy < mask_cy_min or mask_cy > mask_cy_max:
                break

            # Outer dimensions
            mask_xrange_min = 500
            mask_xrange_max = 2000
            mask_xrange = max(np.where(mask)[0])-min(np.where(mask)[0])
            if mask_xrange < mask_xrange_min or mask_xrange > mask_xrange_max:
                break
            
            mask_yrange_min = 500
            mask_yrange_max = 1500
            mask_yrange = max(np.where(mask)[1])-min(np.where(mask)[1])
            if mask_yrange < mask_yrange_min and mask_yrange > mask_yrange_max:
                break

            # Greenness
            mask_green_min = 0.8
            mask_coords = np.array(np.where(mask)).transpose()
            mask_rgb = np.array([img[x, y] for x, y in  mask_coords])
            mask_green = (mask_rgb[:,0] < mask_rgb[:,1]) & (mask_rgb[:,2] < mask_rgb[:,1])
            mask_greenness = np.mean(mask_green)
            if mask_greenness < mask_green_min:
                break

            # Filledness
            fill_rate_min = 0.9
            hull = convex_hull_image(mask)
            fill_rate = np.sum(hull & mask.astype(bool))/np.sum(hull)
            if fill_rate < fill_rate_min:
                break

            # All test passed        
            candidate = {
                'mask': mask,
                'greenness': mask_greenness,
                'fill_rate': fill_rate
            }
            candidates.append(candidate)

    print(len(candidates), 'candidates')

    # Best candidate has highest product of greenness and fill_rate
    if len(candidates) > 0:
        i_winner = np.argmax([(can['greenness']*can['fill_rate']) for can in candidates])
        mask = candidates[i_winner]['mask']
        if visualize:
            plt.figure()
            plt.subplot(121)
            plt.imshow(img)
            plt.subplot(122)
            plt.imshow(mask)
            plt.show()
        return mask

    else:
        print('WARNING: No mask generated.')
        if visualize:
            plt.figure()
            plt.subplot(121)
            plt.imshow(img)
            plt.subplot(122)
            plt.imshow(np.zeroes(img.shape))
            plt.show()
        return None
        

In [ ]:
def prepare_model():
    
    # Select the device for computation
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"using device: {device}")
    
    if device.type == "cuda":
        # use bfloat16 for the entire notebook
        torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
        # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
        if torch.cuda.get_device_properties(0).major >= 8:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
    elif device.type == "mps":
        print(
            "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
            "give numerically different outputs and sometimes degraded performance on MPS. "
            "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
        )
    
    # Prepare model
    sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
    predictor = SAM2ImagePredictor(sam2_model)

    return predictor

In [ ]:
# Input
folder_in_main = '/media/joris/rootfs/home/joris/data/kiwibes2026/20260706kiwibes/bes21-30-joris'


In [ ]:
# Flow
file_in = np.random.choice(glob.glob(f'{folder_in_main}/*/*'))
print(file_in)
img = cv2.imread(file_in)
predictor = prepare_model()
mask = make_mask(img, predictor, visualize = True)


# EXP

In [ ]:
STOP

In [ ]:
# # Prepare output folders

# folder_out = '/tmp/kiwibes2026'
# # if os.path.exists(folder_out):
# #     shutil.rmtree(folder_out)
# os.mkdir(folder_out)

# folder_out_img_mask = f'{folder_out}/img_mask'
# os.mkdir(folder_out_img_mask)

# folder_out_mask = f'{folder_out}/mask'
# os.mkdir(folder_out_mask)

# # file_in = np.random.choice(glob.glob(f'{folder_in_main}/*/*'))
# # for file_in in glob.glob(f'{folder_in_main}/*/*143304*'): 
# # 142950
# # for file_in in np.random.choice(glob.glob(f'{folder_in_main}/*/*'), 10):
# # for file_in in sorted(glob.glob(f'{folder_in_main}/*/*'))[:1]:
# for file_in in sorted(glob.glob(f'{folder_in_main}/*/*'))[:3]:
    
#     print()
#     print(file_in)
    
#     # Load
#     img = cv2.imread(file_in)
    
#     # Generate query points for SAM
#     step = 500
#     queries = list(itertools.product(
#         np.arange(step//2, img.shape[1], step),
#         np.arange(step//2, img.shape[0], step), 
#     ))
    
#     # Apply SAM2
#     predictor.set_image(img)
#     # all_masks = []
#     candidates = []
    
#     for point in queries:
#         input_point = np.array([point])
#         input_label = np.array([1])
#         masks, scores, logits = predictor.predict(
#             point_coords=input_point,
#             point_labels=input_label,
#             multimask_output=True,
#         )
    
#         # Some selection
#         # fill_rate_best = 0
        
#         for mask in masks:
    
#             # Size
#             mask_size_min = 100000
#             mask_size_max = 1000000
#             mask_size = np.sum(mask)
#             if mask_size < mask_size_min or mask_size > mask_size_max:
#                 break
        
#             # Centeredness
#             mask_cx_min = mask.shape[0]*0.25
#             mask_cx_max = mask.shape[0]*0.75
#             mask_cx = np.mean(np.where(mask)[0])
#             if mask_cx < mask_cx_min or mask_cx > mask_cx_max:
#                 break
            
#             mask_cy_min = mask.shape[1]*0.25
#             mask_cy_max = mask.shape[1]*0.75
#             mask_cy = np.mean(np.where(mask)[1])
#             if mask_cy < mask_cy_min or mask_cy > mask_cy_max:
#                 break

#             # Outer dimensions
#             mask_xrange_min = 500
#             mask_xrange_max = 2000
#             mask_xrange = max(np.where(mask)[0])-min(np.where(mask)[0])
#             if mask_xrange < mask_xrange_min or mask_xrange > mask_xrange_max:
#                 break
            
#             mask_yrange_min = 500
#             mask_yrange_max = 1500
#             mask_yrange = max(np.where(mask)[1])-min(np.where(mask)[1])
#             if mask_yrange < mask_yrange_min and mask_yrange > mask_yrange_max:
#                 break

#             # Greenness
#             mask_green_min = 0.8
#             mask_coords = np.array(np.where(mask)).transpose()
#             mask_rgb = np.array([img[x, y] for x, y in  mask_coords])
#             mask_green = (mask_rgb[:,0] < mask_rgb[:,1]) & (mask_rgb[:,2] < mask_rgb[:,1])
#             mask_greenness = np.mean(mask_green)
#             if mask_greenness < mask_green_min:
#                 break

#             # Filledness
#             fill_rate_min = 0.9
#             hull = convex_hull_image(mask)
#             fill_rate = np.sum(hull & mask.astype(bool))/np.sum(hull)
#             if fill_rate < fill_rate_min:
#                 break

#             # All test passed        
#             # all_masks.append(mask)
#             candidate = {
#                 'mask': mask,
#                 'greenness': mask_greenness,
#                 'fill_rate': fill_rate
#             }
#             candidates.append(candidate)

#     print(len(candidates), 'candidates')

#     # Best candidate has highest product of greenness and fill_rate
#     if len(candidates) > 0:
#         i_winner = np.argmax([(can['greenness']*can['fill_rate']) for can in candidates])
#         mask = candidates[i_winner]['mask']
    
#         # Vis
#         img_np = np.asarray(img)
#         if img_np.ndim == 3 and img_np.shape[0] == 3:
#             img_np = np.transpose(img_np, (1, 2, 0))
#         if img_np.dtype != np.uint8:
#             if img_np.max() <= 1.0:
#                 img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
#             else:
#                 img_np = img_np.clip(0, 255).astype(np.uint8)
#         img_pil = Image.fromarray(img_np)
#         stem = Path(file_in).stem
#         mask_np = np.asarray(mask) > 0
#         result = img_np.copy()
#         result[mask_np] = (
#             0.6 * result[mask_np] + 0.4 * np.array([255, 0, 0])
#         ).astype(np.uint8)
#         file_out = f"{folder_out_img_mask}/{stem}.jpg"
#         Image.fromarray(result).save(file_out, quality=95)
#         print(file_out)
    
    
#         # Save mask
#         file_out = f"{folder_out_mask}/{stem}.npy"
#         np.save(file_out, mask)
#         print(file_out)

# 'READY'